<a href="https://colab.research.google.com/github/MazenSherif4/Docking_01/blob/main/04_Cheminfo_crash_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="text-align:center;">
  <img src="https://github.com/MolSSI-Education/iqb-2025/blob/main/images/molssi_main_outline.png?raw=true" style="display: block; margin: 0 auto; max-height:325px;">
</div>


# Molecular Docking using gnina

*This tutorial was written by Jessica Nash (Software Scientist) at The Molecular Sciences Software Institue for the [Cheminformatic-Driven Molecular Docking Workshop](https://pdb101.rcsb.org/news/67d9853eaddf75595bd158f7) held as a Crash Course with the Institute for Quantitative Biomedicine (IQB) and the Protein Data Bank (PDB). Special thanks to Pat Walters and David Koes for feedback, suggestions, and proofreading on this notebook.*

*This notebook is Part 4 of 4 in the notebook series.*

Other notebooks in this series:
1. [Digital Representation of Molecules](https://colab.research.google.com/github/MolSSI-Education/iqb-2025/blob/main/01_Cheminfo_crash_course.ipynb)
2. [Exploring Chemical and Biological Data with BindingDB and the RDKit](https://colab.research.google.com/github/MolSSI-Education/iqb-2025/blob/main/02_Cheminfo_crash_course.ipynb)
3. [Preparing Structures for Docking](https://colab.research.google.com/github/MolSSI-Education/iqb-2025/blob/main/03_Cheminfo_crash_course.ipynb)
4. **Molecular Docking using gnina** (this notebook)

In this notebook, we will perform molecular docking of our original bound ligand along with the ligands we prepared in the last notebook.
Molecular docking is a computational method that predicts the preferred orientation of one molecule (the ligand) when bound to another molecule (the receptor, often a protein).

For docking, we will use a program called [gnina](https://github.com/gnina/gnina). gnina is pronounced `nee-na` (silent g) and is a fork of a software program called smina, which is itself a fork of Autodock Vina.
For those unfamiliar with software development lingo, a "fork" is a copy of a project. Forks may be modified and diverge from the original project.
Autodock Vina is the program we used for docking in last year's PDB Crash Course.

smina was created from AutoDock Vina in order to allow easier set up of docking calculations as well as to allow more customization of the scoring function. smina allows you to set the binding site automatically based on distance from a ligand as well as to define your own scoring functions. gnina builds on that by adding rescoring with convoluational neural networks to improve pose prediction. In this notebook, we will use gnina.

If you find gnina useful and use it in your own research, please be sure to cite the appropriate papers:

Citation
========

**GNINA 1.0: Molecular docking with deep learning** (Primary application citation)  
A McNutt, P Francoeur, R Aggarwal, T Masuda, R Meli, M Ragoza, J Sunseri, DR Koes. *J. Cheminformatics*, 2021  
[link](https://jcheminf.biomedcentral.com/articles/10.1186/s13321-021-00522-2) [PubMed](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8191141/) [ChemRxiv](https://chemrxiv.org/articles/preprint/GNINA_1_0_Molecular_Docking_with_Deep_Learning/13578140)

**Protein–Ligand Scoring with Convolutional Neural Networks**  (Primary methods citation)  
M Ragoza, J Hochuli, E Idrobo, J Sunseri, DR Koes. *J. Chem. Inf. Model*, 2017  
[link](http://pubs.acs.org/doi/full/10.1021/acs.jcim.6b00740) [PubMed](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5479431/) [arXiv](https://arxiv.org/abs/1612.02751)  

**Ligand pose optimization with atomic grid-based convolutional neural networks**  
M Ragoza, L Turner, DR Koes. *Machine Learning for Molecules and Materials NIPS 2017 Workshop*, 2017  
[arXiv](https://arxiv.org/abs/1710.07400)  

**Visualizing convolutional neural network protein-ligand scoring**  
J Hochuli, A Helbling, T Skaist, M Ragoza, DR Koes.  *Journal of Molecular Graphics and Modelling*, 2018  
[link](https://www.sciencedirect.com/science/article/pii/S1093326318301670) [PubMed](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6343664/) [arXiv](https://arxiv.org/abs/1803.02398)

**Convolutional neural network scoring and minimization in the D3R 2017 community challenge**  
J Sunseri, JE King, PG Francoeur, DR Koes.  *Journal of computer-aided molecular design*, 2018  
[link](https://link.springer.com/article/10.1007/s10822-018-0133-y) [PubMed](https://www.ncbi.nlm.nih.gov/pubmed/29992528)

**Three-Dimensional Convolutional Neural Networks and a Cross-Docked Data Set for Structure-Based Drug Design**  
PG Francoeur, T Masuda, J Sunseri, A Jia, RB Iovanisci, I Snyder, DR Koes. *J. Chem. Inf. Model*, 2020  
[link](https://pubs.acs.org/doi/abs/10.1021/acs.jcim.0c00411) [PubMed](https://pubmed.ncbi.nlm.nih.gov/32865404/) [Chemrxiv](https://chemrxiv.org/articles/preprint/3D_Convolutional_Neural_Networks_and_a_CrossDocked_Dataset_for_Structure-Based_Drug_Design/11833323/1)

**Virtual Screening with Gnina 1.0**
J Sunseri, DR Koes D. *Molecules*, 2021
[link](https://www.mdpi.com/1420-3049/26/23/7369) [Preprints](https://www.preprints.org/manuscript/202111.0329/v1)

In [ ]:
# @title Overview
%%html
<style>
div.alert {
    color: #0056b3;
    background-color: #d9edf7;
    border-left: 5px solid #31708f;
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
}
div.alert ul {
    margin: 0.5em 0;
}
div.alert li {
    margin-bottom: 0.5em;
}
</style>

<div class="alert alert-block alert-info">
    <strong>Questions:</strong>
    <ul>
        <li>How can I perform molecular docking using the gnina software?</li>
        <li>What is redocking and how do I perform it?</li>
        <li>What is crossdocking and how do I perform it?</li>
        <li>How are docking results evaluated using scores and Root Mean Square Deviation (RMSD)?</li>
        <li>How do `gnina`'s CNN scores differ from traditional Vina scores in practice?</li>
        <li>How can I dock multiple ligands with gnina?</li>
        <li>How can docking results for multiple compounds be compared?</li>
    </ul>

    <strong>Objectives:</strong>
    <ul>
        <li>Use `gnina` to perform molecular docking for single and multiple ligands.</li>
        <li>Visualize docked poses within the protein binding site.</li>
        <li>Calculate RMSD to evaluate redocking and cross docking accuracy.</li>
        <li>Interpret the different scoring outputs from `gnina`.</li>
    </ul>
</div>

## Set Up
The cells in this section set up the software and files we will need for our calculations.




### Install Python Packages  
1. `useful_rdkit_utils` is a Python package written and maintained by Pat Walters that contains useful RDKit functions. We will use it for the functions `mcs_rmsd` (explained later).
2. `py3Dmol` is used for molecular visualization.
3. The RDKit is a popular cheminiformatics package we will use for processing molecules.


In [ ]:
%%capture
!pip install useful_rdkit_utils py3Dmol rdkit
!apt install openbabel

### Download gnina

We are downloading the pre-compiled binary of gnina. You may also compile gnina yourself by following the directions on the [gnina GitHub repository](https://github.com/gnina/gnina).

In [ ]:
# Download gnina
!wget https://github.com/gnina/gnina/releases/download/v1.3/gnina.fix

--2026-07-10 16:36:06--  https://github.com/gnina/gnina/releases/download/v1.3/gnina.fix
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/45548146/a7090e9d-ca5b-4232-b307-e29a70dbe6d5?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-10T17%3A30%3A11Z&rscd=attachment%3B+filename%3Dgnina.fix&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-10T16%3A29%3A46Z&ske=2026-07-10T17%3A30%3A11Z&sks=b&skv=2018-11-09&sig=A2Tph98%2FqokCKxinfXG2YzlPrk2%2FWNLyrahTSrohAgc%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MzcwNDk2NiwibmJmIjoxNzgzNzAxMzY2LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2luZG93c

In [ ]:
# Make gnina executable
!mv gnina.fix gnina
!chmod +x gnina

### Get Lesson Files

We have stored the files created in the last notebook as a zip file and stored it on GitHub. This cell downloads that file as well as `util.py` which contains a custom utility function for visualizing our ligand and protein.



In [ ]:
%%capture
!wget https://raw.githubusercontent.com/MazenSherif4/Docking_01/main/docking_files.zip
!wget https://raw.githubusercontent.com/MolSSI-Education/iqb-2025/refs/heads/main/util.py

In [ ]:
!unzip -o docking_files.zip

Archive:  docking_files.zip
  inflating: docking_files/ache/best_redocked_coligand_4ey7_pose1.sdf  
  inflating: docking_files/ache/config.txt.txt  
  inflating: docking_files/ache/docked_poses_4ey7_2353.sdf  
  inflating: docking_files/ache/docked_poses_4ey7_fragment.sdf  
  inflating: docking_files/ache/ligand_structures/berberinecpds.sdf  
  inflating: docking_files/ache/ligand_structures/coligand_donepzil.sdf  
  inflating: docking_files/ache/protein_structures/4ey7.pdb  
  inflating: docking_files/ache/protein_structures/4ey7.pdbqt  
  inflating: docking_files/clotting_factor_xa/bestdocked_pose_2353.dwar  
  inflating: docking_files/clotting_factor_xa/bestdocked_pose_berberine.sdf  
  inflating: docking_files/clotting_factor_xa/bestdocked_pose_berZnO.sdf  
  inflating: docking_files/clotting_factor_xa/best_redocked_coligand_pose1.sdf  
  inflating: docking_files/clotting_factor_xa/config.txt.txt  
  inflating: docking_files/clotting_factor_xa/docked_poses_2353(BER).sdf  
  inflati

## Docking with gnina

Molecular Docking involves involves two main stages:

1. Sampling: The algorithm explores many possible positions and orientations (or "poses") of the ligand within the receptor's active site. In AutoDock Vina, smina, and gnina, conformations are generated using Monte Carlo Sampling.
2. Scoring: Each generated pose is evaluated using a scoring function which estimates the binding affinity. Poses are then ranked on these scores. For Vina scores, lower energy scores indicating more favorable interactions.

In addition to the traditional scoring functions avaiable in Vina and smina, gnina adds convolutional neural networks (CNNs) to scoring.  These deep learning models analyze a 3D grid representation of the protein-ligand complex, essentially evaluating a "picture" of the interaction based on atomic densities.

By default, gnina uses results from the CNN for **rescoring**, meaning that poses are initially sampled and scored with the traditional Vina scoring function but re-ranked after sampling using CNN models. You can, however, choose to use the CNN for all scoring, refinement, or not at all (using CNN scoring for refinement or all scoring is more computationally intensive).

For more details see the paper on [gnina v1.0](https://jcheminf.biomedcentral.com/articles/10.1186/s13321-021-00522-2) and [gnina v1.3](https://jcheminf.biomedcentral.com/articles/10.1186/s13321-025-00973-x).

## Redocking the Ligand

Redocking (also called "cognate docking") involves redocking a ligand back into the receptor structure from which the bound pose was experimentally determined.
Redocking is typically done to evaluate how well a docking program's sampling algorithm and scoring function and reproduce a known experimental binding pose.

We will begin our docking journey with gnina by performing a redock of our ligand.

In [ ]:
from util import visualize_poses

v = visualize_poses(
    "docking_files/maoa/protein_structures/2BXS.pdb",
    "docking_files/maoa/ligand_structures/coligand_corgyline.sdf",
)
v.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

You may execute the cell below, and read the following explanation on the input parameters for gnina. gnina works through the command line, so we cannot use in-line comments.

```
./gnina \
  # Specify the receptor structure file (-r).
  # This file (7LME.pdbqt) should be prepared for docking (e.g., with hydrogens added).
  -r docking_files/7LME_all_atom.pdbqt \
  # Specify the ligand structure file (-l) to be docked.
  # This file (Y6J_ideal.pdbqt) contains the 3D coordinates of the ligand.
  -l docking_files/Y6J_ideal.pdbqt \
  # Define the docking search box automatically (--autobox_ligand).
  # The box will be centered around the coordinates of the ligand in the specified file
  # (Y6J_corrected_pose.sdf), which is the known experimental pose in this redocking example.
  # An optional padding (default 4Å) is added.
  --autobox_ligand docking_files/Y6J_corrected_pose.sdf \
  # Specify the output file path (-o) where the resulting docked poses will be saved.
  # The output format will be SDF, containing multiple poses ranked by score.
  -o docking_results/Y6J_docked_e12.sdf \
  # Set the random number generator seed (--seed) to 0.
  # Using a fixed seed makes the docking calculation reproducible.
  --seed 0 \
  # Set the exhaustiveness level (--exhaustiveness) to 12.
  # This controls the number of Monte Carlo chains for the ligand.
  # The default is 8
  --exhaustiveness 16
  ```

  Execute the next cell to run gnina.


In [ ]:
# @title While you Wait: Navigating py3DMol visualizations
%%html
<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>
<div class="purple-box">
   <p> Execute the next cell below to start running
your docking calculation. Then, come back and try these navigation tips for Py3DMol.</p>
    <strong>Py3DMol Visualization Navigation:</strong>
    <table border="1" style="border-collapse: collapse; width: 100%;">
  <thead>
    <tr>
      <th>Movement</th>
      <th>Mouse Input</th>
      <th>Touch Input</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Rotation</td>
      <td>Primary Mouse Button</td>
      <td>Single touch</td>
    </tr>
    <tr>
      <td>Translation</td>
      <td>Middle Mouse Button or Ctrl+Primary</td>
      <td>Triple touch</td>
    </tr>
    <tr>
      <td>Zoom</td>
      <td>Scroll Wheel or Second Mouse Button or Shift+Primary</td>
      <td>Pinch (double touch)</td>
    </tr>
    <tr>
      <td>Slab</td>
      <td>Ctrl+Second</td>
      <td>Not Available</td>
    </tr>
  </tbody>
</table>
</div>

Movement,Mouse Input,Touch Input
Rotation,Primary Mouse Button,Single touch
Translation,Middle Mouse Button or Ctrl+Primary,Triple touch
Zoom,Scroll Wheel or Second Mouse Button or Shift+Primary,Pinch (double touch)
Slab,Ctrl+Second,Not Available


In [ ]:
# Install necessary CUDA 12 libraries for gnina
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cusparse-cu12 nvidia-cublas-cu12 nvidia-curand-cu12 nvidia-cufft-cu12

# Link the library paths to LD_LIBRARY_PATH using %env magic
import os
import nvidia.cuda_runtime
import nvidia.cusparse
import nvidia.cublas
import nvidia.cufft

# Collect all library paths
paths = [
    nvidia.cuda_runtime.__path__[0] + "/lib",
    nvidia.cusparse.__path__[0] + "/lib",
    nvidia.cublas.__path__[0] + "/lib",
    nvidia.cufft.__path__[0] + "/lib",
    "/usr/lib64-nvidia"
]

new_path = ":".join(paths)
%env LD_LIBRARY_PATH=$new_path

env: LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib:/usr/lib64-nvidia


In [ ]:
# 1. Download the static release binary from GitHub
!wget https://github.com/gnina/gnina/releases/download/v1.0.3/gnina -O gnina

# 2. Make the file executable
!chmod +x gnina

# 3. Test if it works
!./gnina --help

--2026-07-10 16:39:27--  https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/45548146/6841601b-545d-4c9f-b4b1-f0eaa8ec7b74?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-10T17%3A30%3A16Z&rscd=attachment%3B+filename%3Dgnina&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-10T16%3A29%3A59Z&ske=2026-07-10T17%3A30%3A16Z&sks=b&skv=2018-11-09&sig=ZXo785YEwtzfY6bnMeY4tSGfQ0UkcM2BwiSCmaMU5Ms%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MzcwNTE2NywibmJmIjoxNzgzNzAxNTY3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2luZG93cy5uZXQifQ.

In [ ]:
!mkdir -p docking_results

# Execute cognate redocking of coligand back into 3CEN
!./gnina \
  -r docking_files/maoa/protein_structures/2BXS.pdbqt \
  -l docking_files/maoa/ligand_structures/coligand_corgyline.sdf \
  --autobox_ligand docking_files/maoa/ligand_structures/coligand_corgyline.sdf \
  -o docking_results/redocked_coligand_2BXS.sdf \
  --seed 0 \
  --exhaustiveness 16

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:e9cb230+   Built Feb 11 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Recommend running with single model (--cnn crossdock_default2018)
or without cnn scoring (--cnn_scoring=none).

Commandline: ./gnina -r docking_files/maoa/protein_structures/2BXS.pdbqt -l docking_files/maoa/ligand_structures/coligand_corgyline.sdf --autobox_ligand docking_files/maoa/ligand_structures/coligand_corgyline.sdf -o docking_results/redocked_coligand_2BXS.sdf --seed 0 --exhaustiveness 16
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************


### Interpreting the output

When `gnina` finishes a docking run, it prints a summary table for the generated poses. This table is sorted by pose rank, with the poses `gnina` determined to be the best at the top.

The columns are the following:

* `mode`: pose rank
* `affinity (kcal/mol)`: the Vina score
* `intra (kcal/mol)`: the ligand's internal strain energy according to the Vina function
* `CNN pose score`: the score from the convolutional neural network predicting pose quality, where higher values closer to 1 indicate higher confidence in the pose's geometric accuracy and are used for ranking.
* `CNN affinity`: The CNN's prediction of binding affinity, expressed in pK units (higher values mean stronger binding, e.g., a predicted score of 9 corresponds to nanomolar (nM) affinity).

Looking at tscores for the redocked Y6J ligand, the table shows the poses ranked by `CNN pose score` (higher is better), with `mode 1` scoring highest (approx. 0.81). Notably, this differs from the ranking by the Vina `affinity` score (lower is better), where `mode 3` is most favorable (-7.51 kcal/mol) but has a much lower `CNN pose score` (approx. 0.49).


### Visualizing the Docked Structures

In the cell below, we use a function called `visualize_docked_structures`, a custom function defined for this workshop that we obtained above when we retreived `util.py`.
This function allows us to view our generated docked structures along with ligand it its original experimentally determined position.

In [ ]:
v = visualize_poses(
    "docking_files/maoa/protein_structures/2BXS.pdb",
    "docking_results/redocked_coligand_2BXS.sdf",
    cognate_file="docking_files/maoa/ligand_structures/coligand_corgyline.sdf",
    animate=False,
)  # Change to True to see an animation of all of the poses
v.show()

### Measuring Root-Mean-Square-Deviation (RMSD)

After generating docked poses, we next need to quantitatively evaluate how close the known reference structure.
The standard metric used for this comparison is the **Root Mean Square Deviation (RMSD)**.
RMSD measures the average distance between corresponding atoms of two molecular structures.
A lower RMSD value indicates greater similarity between the docked pose and the reference structure.
Mathematically, it's calculated as:

$$
RMSD = \sqrt[2]{\frac{1}{N} \sum_{i=1}^{N} \delta_i^2}
$$



where $N$ is the number of corresponding atom pairs being compared, and $\delta_i$ is the Euclidean distance between the $i$-th pair of atoms
In docking studies, a common threshold for considering a docked pose "successful" or accurate is an RMSD below 2 Angstroms compared to the crystal structure.

In this notebook, we will use the `mcs_rmsd` function from the `useful_rdkit_utils` package, written by Pat Walters (a co-instructor of this workshop!).
This function calculates the RMSD, but with a useful modification: it first identifies the **Maximum Common Substructure (MCS)** between the two input molecules using RDKit's `FindMCS` functionality.
It then calculates the RMSD using only the corresponding atoms belonging to this shared substructure.
This approach is particularly valuable when comparing molecules that are similar but not identical, as it focuses the RMSD calculation on the parts of the molecules that match.
While for redocking the original ligand the MCS will typically be the entire molecule, this function can be used later when we compare the poses of different (but similar) docked ligands to the original crystal ligand.



In [ ]:
import useful_rdkit_utils as uru
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

# Silence benign RDKit warning logs for clean terminal output
RDLogger.DisableLog('rdApp.*')

# 1. Load the experimental reference crystal pose (with full sanitization)
cognate = Chem.MolFromMolFile("docking_files/maoa/ligand_structures/coligand_corgyline.sdf")

# 2. Load docked poses without sanitization initially to bypass valence crashes
raw_poses = Chem.SDMolSupplier("docking_results/redocked_coligand_2BXS.sdf", sanitize=False)

print("Pose #\tMatched Atoms\tTrue Full RMSD (Å)\tValidation Status")
print("=" * 65)

for i, raw_pose in enumerate(raw_poses):
    if raw_pose is None:
        continue

    try:
        # ----------------------------------------------------------------------
        # 🛠️ THE CORE FIX: AssignBondOrdersFromTemplate
        # This copies the correct aromaticity and bond orders from your cognate
        # crystal reference onto the docked 3D coordinates, restoring all 33 atoms!
        # ----------------------------------------------------------------------
        clean_pose = AllChem.AssignBondOrdersFromTemplate(cognate, raw_pose)

        # Now compute RMSD across the fully restored molecule
        n_match, rmsd = uru.mcs_rmsd(cognate, clean_pose)

        if rmsd <= 2.0:
            status = "✅ SUCCESS (Valid Experimental Replication)"
        elif rmsd <= 3.0:
            status = "⚠️ MODERATE (Near-Native Deviation)"
        else:
            status = "❌ FAIL (Alternative Binding Mode)"

        print(f"#{i+1}\t{n_match} / 36\t\t{rmsd:.2f}\t\t\t{status}")

    except Exception as e:
        print(f"#{i+1}\tCould not standardize chemical bonds: {e}")

# Re-enable RDKit logging
RDLogger.EnableLog('rdApp.*')

In [ ]:
from rdkit import Chem
from rdkit.Chem import SDWriter
from google.colab import files
import os

input_file = "docking_results/redocked_coligand_2BXS.sdf"
output_file = "best_redocked_coligand_2BXS_pose1.sdf"

# 1. Load the 9 redocked poses without sanitization
supplier = Chem.SDMolSupplier(input_file, sanitize=False)

# 2. Grab only Pose #1 (the very first molecule at index 0)
pose_1 = supplier[0]

if pose_1 is not None:
    # 3. Write just Pose #1 to a clean standalone file
    writer = SDWriter(output_file)
    writer.write(pose_1)
    writer.close()

    print(f"✅ Successfully extracted Pose #1 (1.44 Å match) to: {output_file}")

    # 4. Trigger browser download
    files.download(output_file)

In [ ]:
# @title Exercise
%%html
<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">

<strong>How is docking without the CNN rescoring?</strong>

<p>You can turn off CNN rescoring with gnina by adding <code>--cnn_scoring none</code> to
  your gnina command. Try doing this - make sure to save your results in a new file.

  How does it affect redocking, particularly the measured RMSD score of the docked structures?

</div>

## Docking our Prepared Ligands

In this section, we will dock the ligands we prepared in our previous notebook. Luckily, gnina allows docking of multiple ligands by providing an SDF with your ligands of choice.

In [ ]:
!mkdir -p docking_results

# Execute Virtual Screening of Berberine Library against protein
!./gnina \
  -r docking_files/maoa/protein_structures/2BXS.pdbqt \
  -l docking_files/maoa/ligand_structures/berberinecpds.sdf \
  --autobox_ligand docking_files/maoa/ligand_structures/coligand_corgyline.sdf \
  --autobox_add 4 \
  -o docking_results/berberinecpds_docked_2BXS.sdf \
  --seed 0 \
  --exhaustiveness 16 \
  --min_rmsd_filter 2.0

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:e9cb230+   Built Feb 11 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Recommend running with single model (--cnn crossdock_default2018)
or without cnn scoring (--cnn_scoring=none).

Commandline: ./gnina -r docking_files/nrf2/protein_structures/6TYM.pdbqt -l docking_files/nrf2/ligand_structures/berberinecpds.sdf --autobox_ligand docking_files/nrf2/ligand_structures/coligand_089.sdf --autobox_add 4 -o docking_results/berberinecpds_docked_6TYM.sdf --seed 0 --exhaustiveness 16 --min_rmsd_filter 2.0
*** Open Babel Warning  in ReadMolecule
  Failed to kekulize aromatic bonds in MOL file (title is obj01)

Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|---

In [ ]:
!wget https://raw.githubusercontent.com/MazenSherif4/Docking_01/main/docking_files.zip
!unzip -o docking_results.zip

--2026-07-09 06:51:57--  https://raw.githubusercontent.com/MazenSherif4/Docking_01/main/docking_files.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 429 Too Many Requests
2026-07-09 06:52:00 ERROR 429: Too Many Requests.

unzip:  cannot find or open docking_results.zip, docking_results.zip.zip or docking_results.zip.ZIP.


### Extracting the Scores

gnina stores information about docking poses and their associated scoring metrics directly inside the output .sdf files generated during the simulation. To analyze and compare the results from all our docking runs—specifically your cognate redocked coligand (redocked_coligand_3CEN.sdf) and your virtual screening compound library (berberinecpds_docked_3CEN.sdf)—we need to extract this scoring data from the .sdf files and organize it into a structured Pandas DataFrame table.

We can use RDKit PandasTools to read the molecular structures (poses) and their associated properties (scores) from the output SDF. The SDF will contain multiple poses for the docked ligands, and each pose record has the calculated scores (like `minimizedAffinity`, `CNNscore`, `CNNaffinity`, `CNN_VS`, etc.) stored as data fields. The `CNN_VS` score is the product of `CNNscore` and `CNNaffinity`. We would typically want ligands that score highly for both (and thus have a high `CNN_VS` score.

In [ ]:
from rdkit.Chem import PandasTools
from rdkit.rdBase import BlockLogs
from rdkit import RDLogger
import pandas as pd

# 1. Silence benign RDKit warnings for clean terminal output
RDLogger.DisableLog('rdApp.warning')

# 2. Define the exact scoring tags generated by gnina
score_columns = [
    "minimizedAffinity",
    "CNNscore",
    "CNNaffinity",
    "CNN_VS",
    "CNNaffinity_variance",
]

# 3. Point to your specific project output files
sdf_paths = [
    "docking_results/redocked_coligand_2BXS.sdf",      # Your cognate crystal validation
    "docking_results/berberinecpds_docked_2BXS.sdf",   # Your multi-ligand library screen
]

df_list = []

print("⏳ Loading docked structures and extracting scores...")
for filename in sdf_paths:
    try:
        with BlockLogs():
            # Load the SDF file directly into a Pandas DataFrame
            temp_df = PandasTools.LoadSDF(filename, sanitize=False)

            if temp_df is not None and not temp_df.empty:
                # Add a column to track which file the pose came from
                temp_df["Source_File"] = filename.split("/")[-1]
                df_list.append(temp_df)
            else:
                print(f"⚠️ Warning: No poses loaded from {filename}")
    except Exception as e:
        print(f"❌ Could not load {filename}: {e}")

if not df_list:
    raise ValueError("No docking data could be loaded! Check your file paths.")

# 4. Combine all runs into a single master DataFrame
combo_df = pd.concat(df_list, ignore_index=True)

# 5. PandasTools reads all SDTags as strings; convert score columns to float
for col in score_columns:
    if col in combo_df.columns:
        combo_df[col] = combo_df[col].astype(float)

# 6. Ensure ID column exists for deduplication
if "ID" not in combo_df.columns:
    combo_df["ID"] = combo_df["_Name"] if "_Name" in combo_df.columns else [f"Mol_{i}" for i in range(len(combo_df))]

# 7. Calculate CNN_VS if gnina didn't explicitly output the tag
if "CNN_VS" not in combo_df.columns or combo_df["CNN_VS"].isnull().all():
    combo_df["CNN_VS"] = combo_df["CNNscore"] * combo_df["CNNaffinity"]

print(f"✅ Successfully extracted {len(combo_df)} total poses across all docking runs!")

# ==============================================================================
# 🏆 CONSENSUS TRIAGE BY CNN_VS
# Rank all poses by their CNN_VS product score (descending)
# Keep only the #1 top-scoring pose per molecule ID
# ==============================================================================

top_leads = (
    combo_df.sort_values(by="CNN_VS", ascending=False)
    .drop_duplicates(subset=["ID"])
    .reset_index(drop=True)
)

print("\n🎯 TOP LEAD COMPOUNDS RANKED BY CNN_VS (CNNscore × CNNaffinity):")
display_cols = ["ID", "Source_File", "CNN_VS", "CNNscore", "CNNaffinity", "minimizedAffinity"]
available_cols = [col for col in display_cols if col in top_leads.columns]
display(top_leads[available_cols].head(15))

# Export the master extracted score table to CSV
combo_df.to_csv("docking_results/all_extracted_scores_3CEN.csv", index=False)
top_leads.to_csv("docking_results/top_ranked_leads_by_CNN_VS.csv", index=False)
print("\n📁 Saved master tables to docking_results/all_extracted_scores_3CEN.csv and top_ranked_leads_by_CNN_VS.csv")

RDLogger.EnableLog('rdApp.warning')

⏳ Loading docked structures and extracting scores...
✅ Successfully extracted 27 total poses across all docking runs!

🎯 TOP LEAD COMPOUNDS RANKED BY CNN_VS (CNNscore × CNNaffinity):


,ID,Source_File,CNN_VS,CNNscore,CNNaffinity,minimizedAffinity
0,obj01,redocked_coligand_6TYM.sdf,7.546572,0.989803,7.624314,-12.34216
1,2353,berberinecpds_docked_6TYM.sdf,3.654344,0.649627,5.625294,-7.06973
2,Fragment,berberinecpds_docked_6TYM.sdf,2.288693,0.426939,5.360704,-5.86935



📁 Saved master tables to docking_results/all_extracted_scores_3CEN.csv and top_ranked_leads_by_CNN_VS.csv


In [ ]:
from rdkit import Chem
from rdkit.Chem import SDWriter
import os

# Define the input file (pointing to the correct results directory) and output filenames
input_sdf = "docking_results/berberinecpds_docked_2BXS.sdf"
output_files = {
    "2353": "docked_poses_2BXS_2353.sdf",
    "fragment": "docked_poses_2BXS_fragment.sdf"
}

# Create writers for the output files
writers = {name: SDWriter(filename) for name, filename in output_files.items()}

# Read the master multi-ligand SDF file
supplier = Chem.SDMolSupplier(input_sdf, sanitize=False)

extracted_counts = {"2353": 0, "fragment": 0}

for mol in supplier:
    if mol is None:
        continue

    # Get the compound identifier
    mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else ""
    mol_id = mol.GetProp("ID") if mol.HasProp("ID") else ""

    # Check and write to the corresponding file
    for target_key in writers.keys():
        if target_key.lower() in mol_name.lower() or target_key.lower() in mol_id.lower():
            writers[target_key].write(mol)
            extracted_counts[target_key] += 1

# Close the writers to save the files properly
for writer in writers.values():
    writer.close()

# Print confirmation
for target_key, count in extracted_counts.items():
    print(f"✅ Extracted {count} poses for '{target_key}' -> Saved to: {output_files[target_key]}")

✅ Extracted 9 poses for '2353' -> Saved to: docked_poses_6TYM_2353.sdf
✅ Extracted 9 poses for 'fragment' -> Saved to: docked_poses_6TYM_fragment.sdf


[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:24:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [ ]:
from google.colab import files

# Download the 2353 poses
files.download("docked_poses_2BXS_2353.sdf")

# Download the fragment poses
files.download("docked_poses_2BXS_fragment.sdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title Key Points
%%html
<style>
div.green-note {
    color: #155724; /* Dark green for text */
    background-color: #d4edda; /* Light green background */
    border-left: 5px solid #28a745; /* Bright green border */
    padding: 0.5em;
    font-size: 1.25em; /* Consistent with text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean and modern font */
}
div.green-note ul {
    margin: 0.5em 0; /* Space around the list */
}
div.green-note li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="green-note">
    <strong>Key Points:</strong>
    <ul>
        <li><code>gnina</code> is used for molecular docking via the command line, requiring prepared receptor (<code>-r</code>) and ligand (<code>-l</code>) files, defining a search box (e.g., <code>--autobox_ligand</code>), and specifying an output file (<code>-o</code>).</li>
        <li>Docking results are evaluated using scoring functions (<code>minimizedAffinity</code> from Vina, <code>CNNscore</code> and <code>CNNaffinity</code> from the neural network) and Root Mean Square Deviation (RMSD) to measure geometric similarity to a known pose.</li>
        <li><code>gnina</code>'s default <code>CNNscore</code> ranks poses based on predicted geometric accuracy (likelihood of low RMSD) and often differs from rankings based on the Vina affinity score (<code>minimizedAffinity</code>).</li>
        <li>Redocking assesses reproducibility against a known structure, while cross-docking (using a different but related receptor structure) tests robustness, requiring protein alignment for meaningful RMSD comparison.</li>
        <li>Multiple ligands can be docked efficiently by providing a multi-molecule SDF file to <code>gnina</code>, and results can be compiled and analyzed using RDKit and Pandas in Python.</li>
    </ul>
</div>